<a href="https://colab.research.google.com/github/Sanjarbek9706/jaydariGPT/blob/main/jaydari_gpt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
# Kerakli Python kutubxonalarini Colab muhitiga o'rnatamiz.
# transformers — tayyor tokenizer va til modellarini yuklash uchun.
# torch — tensorlar va CPU/GPU hisoblashlari bilan ishlash uchun.
# bitsAndBytes — model og'irliklarini kamroq bitda saqlab ishlatish uchun.
# ! terminal buyrug'ini bajaradi; pip install paketlarni o'rnatadi.
!pip install transformers torch bitsAndBytes

In [12]:
# AutoTokenizer — tanlangan modelga mos tokenizerni avtomatik yuklaydi.
# BitsAndBytesConfig — modelni kvantlash sozlamalarini tayyorlaydi.
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
# PyTorch kutubxonasini torch nomi bilan ishlatamiz.
import torch

In [13]:
# Hugging Face'dagi model manzili: tashkilot / model nomi.
model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
# Modelga mos lug'at va matnni tokenlarga ajratish qoidalarini yuklaymiz.
# Bu qator javob yaratadigan model og'irliklarini yuklamaydi.
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Tokenizerning barcha maxsus tokenlarini ro'yxat qilib chiqaradi.
print('Special tokens:', tokenizer.all_special_tokens)
print('Special tokens:', tokenizer.special_tokens_map)
print('Vocab size:', tokenizer.vocab_size)

# Kvantlash — model og'irliklarini kamroq bit bilan ifodalab, xotira sarfini kamaytirish. 4k => 1080p
# Hozir faqat sozlama obyektini yaratamiz; model hali kvantlanmaydi.
bnb_config = BitsAndBytesConfig(
    # Qo'llab-quvvatlanadigan qatlam og'irliklarini 4-bit ko'rinishda yuklash.
    load_in_4bit=True,
    # Kvantlash koeffitsiyentlarini ham kvantlab, xotirani yana tejash.
    bnb_4bit_use_double_quant=True,
    # NF4 — og'irliklar uchun ishlatiladigan 4-bit sonlar formati.
    bnb_4bit_quant_type='nf4',
    # Hisoblashdagi sonlar turi BF16; bu 4-bit saqlash formatidan alohida.
    # Uning qurilmada qo'llanishi GPU va kutubxona imkoniyatlariga bog'liq.
    bnb_4bit_compute_dtype=torch.bfloat16
)

bnb_config
# Modelni yuklashda quantization_config=bnb_config berilsa, sozlamalar qo'llanadi.


model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map='auto', # GPU, CPU
    # dtype=torch.bfloat16
)

Special tokens: ['<s>', '</s>', '<unk>']
Special tokens: {'bos_token': '<s>', 'eos_token': '</s>', 'unk_token': '<unk>', 'pad_token': '</s>'}
Vocab size: 32000


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [23]:
# Before Fine-tuning
prompt = "Explain what a tokenizer is?"
# prompt = "A tokenizer is a tool in natural language processing that"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device) # GPU, CPU

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=True,
        temperature=0.7
    )

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))

# print(model)
first_block = model.model.layers[0]
print('first_block:', first_block)
print('==================')
print('first_block.self_attn:', first_block.self_attn)
print('==================')
print(model.config)



[transformers] Both `max_new_tokens` (=80) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain what a tokenizer is? How do they differ from one another, and what are their most common uses?
first_block: LlamaDecoderLayer(
  (self_attn): LlamaAttention(
    (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
    (k_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (v_proj): Linear4bit(in_features=2048, out_features=256, bias=False)
    (o_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  )
  (mlp): LlamaMLP(
    (gate_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (up_proj): Linear4bit(in_features=2048, out_features=5632, bias=False)
    (down_proj): Linear4bit(in_features=5632, out_features=2048, bias=False)
    (act_fn): SiLUActivation()
  )
  (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
  (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
)
first_block.self_attn: LlamaAttention(
  (q_proj): Linear4bit(in_features=2048, out_features=2048, bias=False)
  (k_proj): Linear4bit(

In [24]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

total_params = count_parameters(model)
print(f"Total parameters (including frozen 4-bit): {total_params:,}")

Total parameters (including frozen 4-bit): 615,606,272
